<a href="https://colab.research.google.com/github/Eliaum/formulaire/blob/main/classification-plantvillage/Modele_InceptionV3_et_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Détection de maladies des plantes : InceptionV3 vs MobileNetV2

**Objectif.** Classer des photos de feuilles de poivron, de pomme de terre et de tomate du jeu de données
[PlantVillage](https://arxiv.org/abs/1511.08060) (15 classes, saines ou malades). On compare deux réseaux
pré-entraînés sur ImageNet, adaptés par **apprentissage par transfert** :

- **InceptionV3** : réseau plus lourd, plus précis en général ;
- **MobileNetV2** : réseau léger, conçu pour les appareils mobiles.

**Méthode.**
1. Inventaire et contrôle des images, puis découpage **stratifié** en jeux d'entraînement, de validation et de
   test (70 / 15 / 15 %). Les fichiers ne sont jamais déplacés ni modifiés.
2. Prétraitement identique à celui des poids ImageNet (pixels ramenés dans [-1, 1]) et augmentation de données
   intégrée au modèle.
3. Entraînement en deux phases : couche de classification seule (réseau de base gelé), puis affinage des
   dernières couches du réseau de base avec un taux d'apprentissage faible.
4. Évaluation sur le **jeu de test**, jamais vu pendant l'entraînement : exactitude, F1 macro, rapport par
   classe, matrice de confusion, taille et vitesse des modèles.

**Exécution.** Sur Google Colab, choisir un GPU (*Exécution › Modifier le type d'exécution*), adapter la
cellule de paramètres, puis *Exécution › Tout exécuter*. Les modèles, métriques et figures sont écrits dans
`OUTPUT_DIR`.

## 1. Paramètres

In [ ]:
# Cellule étiquetée « parameters » : papermill peut en remplacer les valeurs.

# Données : un dossier contenant un sous-dossier par classe (noms de dossiers PlantVillage d'origine).
DATA_DIR = "/content/drive/My Drive/Datasets/PlantVillage"
# Archive utilisée si DATA_DIR n'existe pas ; elle est extraite sur le disque local de Colab.
DATA_ZIP = "/content/drive/My Drive/archive.zip"
LOCAL_DATA_DIR = "/content/plantvillage"
# Copier les images de Drive sur le disque local : lecture beaucoup plus rapide pendant l'entraînement.
COPY_TO_LOCAL = True
# Dossier des résultats (modèles, métriques, figures) ; sur Drive, il survit à la session Colab.
OUTPUT_DIR = "/content/drive/My Drive/plantvillage_resultats"

MODEL_NAMES = ["InceptionV3", "MobileNetV2"]
IMAGE_SIZES = {"InceptionV3": 299, "MobileNetV2": 224}  # tailles d'entrée natives des deux réseaux
WEIGHTS = "imagenet"

VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
MIN_IMAGES_PER_CLASS = 10  # une classe plus petite est écartée (avec un avertissement)
MAX_IMAGES_PER_CLASS = None  # un entier pour un essai rapide sur un sous-échantillon
VERIFY_IMAGES = True  # écarte les fichiers illisibles

BATCH_SIZE = 32
HEAD_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
FINE_TUNE_FRACTION = 0.25  # part des dernières couches du réseau de base dégelées pendant l'affinage
EARLY_STOPPING_PATIENCE = 3
SEED = 42

## 2. Environnement et reproductibilité

In [ ]:
import json
import platform
import shutil
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

IN_COLAB = "google.colab" in sys.modules
keras.utils.set_random_seed(SEED)  # graines de Python, NumPy et TensorFlow

print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | Keras {keras.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU :", ", ".join(gpu.name for gpu in gpus))
else:
    print("Aucun GPU détecté : l'entraînement complet sera très lent.")

## 3. Accès aux données

Le notebook ne télécharge pas le jeu de données : il lit un dossier (`DATA_DIR`) ou une archive (`DATA_ZIP`).
Sur Colab, Google Drive est monté automatiquement si l'un des chemins s'y trouve.

In [ ]:
def on_drive(path):
    return str(path).startswith("/content/drive")


if IN_COLAB and any(on_drive(p) for p in (DATA_DIR, DATA_ZIP, OUTPUT_DIR)):
    from google.colab import drive

    drive.mount("/content/drive")

if Path(DATA_DIR).is_dir():
    data_root = Path(DATA_DIR)
    if COPY_TO_LOCAL and on_drive(data_root):
        local = Path(LOCAL_DATA_DIR)
        if not local.is_dir():
            print(f"Copie de {data_root} vers {local} (une seule fois par session)…")
            shutil.copytree(data_root, local)
        data_root = local
elif DATA_ZIP and Path(DATA_ZIP).is_file():
    data_root = Path(LOCAL_DATA_DIR)
    if not data_root.is_dir():
        print(f"Extraction de {DATA_ZIP} vers {data_root}…")
        with zipfile.ZipFile(DATA_ZIP) as archive:
            archive.extractall(data_root)
else:
    raise FileNotFoundError(
        f"Données introuvables : ni le dossier {DATA_DIR!r} ni l'archive {DATA_ZIP!r} n'existent. "
        "Placez le jeu PlantVillage (un sous-dossier par classe) à l'un de ces emplacements, "
        "ou modifiez la cellule de paramètres."
    )

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print("Données  :", data_root)
print("Résultats:", output_dir)

## 4. Inventaire et contrôle des images

Chaque image reçoit pour étiquette le nom de son dossier parent. Les doublons (même classe et même nom de
fichier) sont ignorés : cela neutralise une copie imbriquée du jeu (`PlantVillage/PlantVillage`) ou des
dossiers `train/` et `val/` laissés par une version précédente de ce notebook.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
EXPECTED_CLASSES = [
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy",
]


def list_images(root):
    seen, rows = set(), []
    for path in sorted(Path(root).rglob("*")):
        if path.suffix.lower() in IMAGE_EXTENSIONS and path.is_file():
            key = (path.parent.name, path.name)
            if key not in seen:
                seen.add(key)
                rows.append({"path": str(path), "label": path.parent.name})
    return pd.DataFrame(rows, columns=["path", "label"])


def is_readable_image(path):
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except (OSError, SyntaxError, ValueError):
        return False


images = list_images(data_root)
if images.empty:
    raise ValueError(f"Aucune image ({', '.join(sorted(IMAGE_EXTENSIONS))}) trouvée dans {data_root}.")
print(f"{len(images)} images trouvées.")

if VERIFY_IMAGES:
    readable = images["path"].map(is_readable_image)
    print(f"{int((~readable).sum())} image(s) illisible(s) écartée(s).")
    images = images[readable]

counts = images["label"].value_counts().sort_index()
too_small = counts[counts < MIN_IMAGES_PER_CLASS]
missing = sorted(set(EXPECTED_CLASSES) - set(counts.index))
if len(too_small):
    print(
        f"⚠️ Classes écartées (moins de {MIN_IMAGES_PER_CLASS} images) : "
        f"{ {label: int(n) for label, n in too_small.items()} }"
    )
if missing:
    print(f"⚠️ {len(missing)} classe(s) PlantVillage attendue(s) absente(s) : {missing}")

images = images[images["label"].isin(counts[counts >= MIN_IMAGES_PER_CLASS].index)]
if MAX_IMAGES_PER_CLASS:
    images = images.sample(frac=1, random_state=SEED).groupby("label").head(MAX_IMAGES_PER_CLASS)

class_names = sorted(images["label"].unique())
if len(class_names) < 2:
    raise ValueError("Il faut au moins deux classes contenant des images pour entraîner un classifieur.")
label_ids = {name: i for i, name in enumerate(class_names)}
images = images.assign(label_id=images["label"].map(label_ids)).reset_index(drop=True)

counts = images["label"].value_counts().reindex(class_names)
print(
    f"{len(images)} images retenues, {len(class_names)} classes ; "
    f"rapport entre la plus grande et la plus petite classe : {counts.max() / counts.min():.1f}."
)
counts.plot.barh(figsize=(8, 0.35 * len(class_names) + 1), title="Nombre d'images par classe")
plt.tight_layout()
plt.show()

In [ ]:
# Une image par classe.
examples = images.groupby("label").head(1)
columns = 5
rows = int(np.ceil(len(examples) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 3.2 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (_, row) in zip(axes.flat, examples.iterrows(), strict=False):
    with Image.open(row["path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row["label"].replace("_", " ").strip(), fontsize=8)
plt.tight_layout()
plt.show()

## 5. Découpage entraînement / validation / test

Le découpage est **stratifié** (mêmes proportions de classes dans chaque jeu) et reproductible (`SEED`).
- Le jeu de **validation** sert à l'arrêt anticipé et à l'ajustement du taux d'apprentissage.
- Le jeu de **test** n'intervient qu'à l'évaluation finale.

Le déséquilibre entre classes est compensé par des poids de classe dans la fonction de perte.

In [ ]:
train_df, holdout_df = train_test_split(
    images, test_size=VAL_FRACTION + TEST_FRACTION, stratify=images["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    holdout_df,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=holdout_df["label"],
    random_state=SEED,
)

split_counts = (
    pd.DataFrame(
        {
            "entraînement": train_df["label"].value_counts(),
            "validation": val_df["label"].value_counts(),
            "test": test_df["label"].value_counts(),
        }
    )
    .reindex(class_names)
    .fillna(0)
    .astype(int)
)
split_counts.loc["Total"] = split_counts.sum()
display(split_counts)

weights = compute_class_weight("balanced", classes=np.arange(len(class_names)), y=train_df["label_id"])
class_weight = {i: float(w) for i, w in enumerate(weights)}

# Traçabilité : découpage et classes enregistrés avec les résultats.
pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")]).assign(
    path=lambda df: df["path"].map(lambda p: str(Path(p).relative_to(data_root)))
).to_csv(output_dir / "decoupage.csv", index=False)
(output_dir / "classes.json").write_text(json.dumps(class_names, ensure_ascii=False, indent=2))

## 6. Chargement des images (`tf.data`)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label, image_size):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(image, (image_size, image_size))  # float32, valeurs dans [0, 255]
    return image, label


def make_dataset(df, image_size, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((df["path"].to_numpy(), df["label_id"].to_numpy()))
    if training:
        dataset = dataset.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda p, y: load_image(p, y, image_size), num_parallel_calls=AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 7. Modèles

Architecture commune :

`image (0–255)` → augmentation (entraînement seulement) → mise à l'échelle dans [-1, 1] → réseau de base
ImageNet → moyenne globale → dropout → couche dense softmax.

La mise à l'échelle `Rescaling(1/127.5, offset=-1)` reproduit exactement `preprocess_input` d'InceptionV3 et
de MobileNetV2. Comme elle fait partie du modèle, le modèle sauvegardé s'utilise directement sur des images
brutes.

In [ ]:
BACKBONES = {"InceptionV3": keras.applications.InceptionV3, "MobileNetV2": keras.applications.MobileNetV2}


def build_model(name, num_classes, image_size):
    inputs = keras.Input(shape=(image_size, image_size, 3), name="image")
    augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=SEED),
            layers.RandomRotation(0.1, seed=SEED),
            layers.RandomZoom(0.1, seed=SEED),
        ],
        name="augmentation",
    )
    x = augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="pretraitement")(x)
    backbone = BACKBONES[name](include_top=False, weights=WEIGHTS, input_shape=(image_size, image_size, 3))
    backbone.trainable = False
    x = backbone(x, training=False)  # BatchNorm toujours en mode inférence, y compris à l'affinage
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3, seed=SEED)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classes")(x)
    return keras.Model(inputs, outputs, name=name), backbone

## 8. Entraînement en deux phases

1. **Tête seule** : le réseau de base est gelé, seule la nouvelle couche de classification apprend
   (taux d'apprentissage `HEAD_LEARNING_RATE`).
2. **Affinage** : les `FINE_TUNE_FRACTION` dernières couches du réseau de base sont dégelées, hors
   BatchNormalization, avec un taux d'apprentissage faible (`FINE_TUNE_LEARNING_RATE`).

À chaque phase :
- l'arrêt anticipé et la réduction du taux d'apprentissage surveillent la perte de validation ;
- le meilleur modèle est sauvegardé au format `.keras`.

Le modèle retenu est celui dont la perte de validation est la plus basse, toutes phases confondues.

In [ ]:
def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


def make_callbacks(checkpoint_path, best_val_loss=None):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=max(1, EARLY_STOPPING_PATIENCE - 1), min_lr=1e-7
        ),
        keras.callbacks.ModelCheckpoint(
            str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            initial_value_threshold=best_val_loss,
        ),
    ]


def train(name):
    image_size = IMAGE_SIZES[name]
    train_ds = make_dataset(train_df, image_size, training=True)
    val_ds = make_dataset(val_df, image_size)
    model, backbone = build_model(name, len(class_names), image_size)
    checkpoint = output_dir / f"{name}.keras"

    # Phase 1 : tête seule.
    compile_model(model, HEAD_LEARNING_RATE)
    head = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=make_callbacks(checkpoint),
        verbose=2,
    )
    model.set_weights(keras.models.load_model(checkpoint).get_weights())  # meilleur état de la phase 1

    # Phase 2 : affinage des dernières couches.
    backbone.trainable = True
    first_trainable = int(len(backbone.layers) * (1 - FINE_TUNE_FRACTION))
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= first_trainable and not isinstance(layer, layers.BatchNormalization)
    compile_model(model, FINE_TUNE_LEARNING_RATE)
    fine = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=len(head.epoch),
        epochs=len(head.epoch) + FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=make_callbacks(checkpoint, min(head.history["val_loss"])),
        verbose=2,
    )

    history = pd.concat(
        [
            pd.DataFrame(head.history).assign(phase="tête"),
            pd.DataFrame(fine.history).assign(phase="affinage"),
        ],
        ignore_index=True,
    )
    history.insert(0, "epoch", np.arange(1, len(history) + 1))
    return keras.models.load_model(checkpoint), history


trained, histories, training_minutes = {}, {}, {}
for name in MODEL_NAMES:
    print(f"\n========== {name} ==========")
    start = time.perf_counter()
    trained[name], histories[name] = train(name)
    training_minutes[name] = (time.perf_counter() - start) / 60
    histories[name].to_csv(output_dir / f"{name}_historique.csv", index=False)
    print(f"{name} : {training_minutes[name]:.1f} min d'entraînement.")

In [ ]:
fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(12, 3.8 * len(MODEL_NAMES)), squeeze=False)
for row, name in zip(axes, MODEL_NAMES, strict=False):
    history = histories[name]
    fine_tune_start = history.loc[history["phase"] == "affinage", "epoch"].min()
    for ax, metric, title in zip(row, ["accuracy", "loss"], ["exactitude", "perte"], strict=False):
        ax.plot(history["epoch"], history[metric], marker="o", label="entraînement")
        ax.plot(history["epoch"], history[f"val_{metric}"], marker="o", label="validation")
        if not np.isnan(fine_tune_start):
            ax.axvline(fine_tune_start - 0.5, color="grey", linestyle="--", label="début de l'affinage")
        ax.set(title=f"{name} – {title}", xlabel="époque")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.legend()
plt.tight_layout()
plt.savefig(output_dir / "courbes_apprentissage.png", dpi=150)
plt.show()

## 9. Évaluation sur le jeu de test

In [ ]:
def inference_ms_per_image(model, dataset):
    batch = next(iter(dataset))[0]
    model.predict_on_batch(batch)  # échauffement
    start = time.perf_counter()
    model.predict_on_batch(batch)
    return (time.perf_counter() - start) * 1000 / len(batch)


y_true = test_df["label_id"].to_numpy()
rows, predictions = [], {}
for name in MODEL_NAMES:
    model = trained[name]
    test_ds = make_dataset(test_df, IMAGE_SIZES[name])
    probabilities = model.predict(test_ds, verbose=0)
    predictions[name] = probabilities
    y_pred = probabilities.argmax(axis=1)
    rows.append(
        {
            "modèle": name,
            "exactitude": accuracy_score(y_true, y_pred),
            "F1 macro": f1_score(y_true, y_pred, average="macro"),
            "F1 pondéré": f1_score(y_true, y_pred, average="weighted"),
            "paramètres (M)": model.count_params() / 1e6,
            "poids float32 (Mo)": model.count_params() * 4 / 1e6,
            "entraînement (min)": training_minutes[name],
            "inférence (ms/image)": inference_ms_per_image(model, test_ds),
        }
    )
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
    (output_dir / f"{name}_rapport_test.txt").write_text(report)
    print(f"========== {name} : rapport par classe (jeu de test, {len(test_df)} images) ==========")
    print(report)

comparison = pd.DataFrame(rows).set_index("modèle")
comparison.to_csv(output_dir / "comparaison_test.csv")
display(comparison.style.format("{:.3f}"))

In [ ]:
def short_name(label):
    """Nom de classe abrégé pour les axes des figures."""
    for prefix, short in [
        ("Pepper__bell___", "Poivron "),
        ("Potato___", "Pomme de terre "),
        ("Tomato__Tomato_", "Tomate "),
        ("Tomato__", "Tomate "),
        ("Tomato_", "Tomate "),
    ]:
        label = label.replace(prefix, short)
    return " ".join(label.replace("_", " ").split())


fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(9 * len(MODEL_NAMES), 8), squeeze=False)
for ax, name in zip(axes[0], MODEL_NAMES, strict=False):
    matrix = confusion_matrix(
        y_true, predictions[name].argmax(axis=1), labels=np.arange(len(class_names)), normalize="true"
    )
    ConfusionMatrixDisplay(matrix, display_labels=[short_name(c) for c in class_names]).plot(
        ax=ax, cmap="Blues", values_format=".2f", colorbar=False, xticks_rotation=90
    )
    ax.set(
        title=f"{name} – matrice de confusion normalisée (test)",
        xlabel="classe prédite",
        ylabel="classe réelle",
    )
plt.tight_layout()
plt.savefig(output_dir / "matrices_confusion.png", dpi=150)
plt.show()

## 10. Exemples de prédictions et utilisation du modèle

In [ ]:
def predict_image(path, model_name, top_k=3):
    """Renvoie les top_k classes prédites pour une image, avec leur probabilité."""
    image_size = IMAGE_SIZES[model_name]
    image, _ = load_image(tf.constant(str(path)), 0, image_size)
    probabilities = trained[model_name].predict(image[tf.newaxis, ...], verbose=0)[0]
    best = probabilities.argsort()[::-1][:top_k]
    return [(class_names[i], float(probabilities[i])) for i in best]


best_model = comparison["F1 macro"].idxmax()
predicted = predictions[best_model].argmax(axis=1)
errors = np.flatnonzero(predicted != y_true)
shown = errors[:8] if len(errors) else np.arange(min(8, len(test_df)))
print(f"Modèle retenu : {best_model} — {len(errors)} erreur(s) sur {len(test_df)} images de test.")

fig, axes = plt.subplots(1, len(shown), figsize=(3 * len(shown), 3.6), squeeze=False)
for ax, i in zip(axes[0], shown, strict=False):
    with Image.open(test_df.iloc[i]["path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"vrai : {class_names[y_true[i]]}\nprédit : {class_names[predicted[i]]}", fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Exemple d'appel :", predict_image(test_df.iloc[0]["path"], best_model))

In [ ]:
# Paramètres et résultats enregistrés pour la traçabilité.
summary = {
    "tensorflow": tf.__version__,
    "parametres": {
        "MODEL_NAMES": MODEL_NAMES,
        "IMAGE_SIZES": IMAGE_SIZES,
        "WEIGHTS": WEIGHTS,
        "VAL_FRACTION": VAL_FRACTION,
        "TEST_FRACTION": TEST_FRACTION,
        "BATCH_SIZE": BATCH_SIZE,
        "HEAD_EPOCHS": HEAD_EPOCHS,
        "FINE_TUNE_EPOCHS": FINE_TUNE_EPOCHS,
        "HEAD_LEARNING_RATE": HEAD_LEARNING_RATE,
        "FINE_TUNE_LEARNING_RATE": FINE_TUNE_LEARNING_RATE,
        "FINE_TUNE_FRACTION": FINE_TUNE_FRACTION,
        "SEED": SEED,
    },
    "images": {"entrainement": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "classes": class_names,
    "resultats_test": comparison.reset_index().to_dict(orient="records"),
}
(output_dir / "resultats.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print("Fichiers produits :", sorted(p.name for p in output_dir.iterdir()))

## 11. Lecture des résultats et limites

- **Mesure de référence.** Le jeu de test reste le seul indicateur de généralisation. Les scores de
  validation sont optimistes, car ils ont servi à choisir le meilleur état de chaque modèle.
- **Déséquilibre des classes.** Le *F1 macro* donne le même poids à toutes les classes. Il est donc plus
  exigeant que l'exactitude. Les petites classes sont à examiner dans le rapport par classe et dans la matrice
  de confusion.
- **Conditions de prise de vue.** Les photos PlantVillage sont homogènes. Rien ne garantit qu'un modèle
  performant sur ce jeu le soit sur des photos prises au champ ; il faudrait le vérifier sur des images
  indépendantes.
- **Variabilité.** Chaque modèle n'est entraîné qu'une fois. Pour conclure qu'une architecture est
  meilleure que l'autre, il faudrait répéter l'expérience avec plusieurs graines et comparer la dispersion
  des scores.
- **Compromis.** MobileNetV2 est plus petit et plus rapide, ce qui le destine au mobile. InceptionV3 vise la
  précision. Le tableau de comparaison chiffre ce compromis.